# 02 — Feature Engineering

Construct time-lag features, rolling aggregates, categorical encodings, interaction terms, and apply leakage guards.

In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from src.features.engineer import LeakageSafeFeatureEngineer

train_df = pd.read_csv('../data/synthetic/loan_monthly_performance_train.csv')
test_df  = pd.read_csv('../data/synthetic/loan_monthly_performance_test.csv')
for col in ['reporting_month', 'origination_month']:
    for df in [train_df, test_df]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col]).dt.to_period('M')
print(f'Train shape before FE: {train_df.shape}')


## Fit & Transform

In [ ]:
fe = LeakageSafeFeatureEngineer()
X_train = fe.fit_transform(train_df, is_train=True)
X_test  = fe.transform(test_df)
print(f'Train after FE: {X_train.shape}')
print(f'Test  after FE: {X_test.shape}')


## Feature Manifest (top features by type)

In [ ]:
import joblib
manifest_path = '../models/feature_engineering/feature_manifest.csv'
try:
    manifest = pd.read_csv(manifest_path)
    print(manifest.groupby('transformation').size().to_string())
    print(f'\nTotal features: {len(manifest)}')
    manifest.head(20)
except FileNotFoundError:
    print('Manifest not found — run full pipeline first')
    cols = pd.DataFrame({'name': X_train.columns})
    print(cols.head(20).to_string(index=False))


## Rolling Feature Distributions

In [ ]:
roll_cols = [c for c in X_train.columns if 'rollmean' in c or 'lag' in c][:6]
fig, axes = plt.subplots(1, len(roll_cols), figsize=(3*len(roll_cols), 3))
if len(roll_cols) == 1: axes = [axes]
for ax, col in zip(axes, roll_cols):
    X_train[col].hist(bins=30, ax=ax, color='teal', edgecolor='white')
    ax.set_title(col.replace('_rollmean_', '\nrollmean_'), fontsize=7)
plt.suptitle('Rolling Feature Distributions', y=1.02)
plt.tight_layout()
plt.savefig('../reports/rolling_features.png', dpi=150)
plt.show()


## Leakage Guard Verification

In [ ]:
# Confirm target columns are not in the feature matrix
target_cols = ['next_3m_delinquency_flag','next_6m_delinquency_flag',
               'next_12m_default_flag','next_12m_prepayment_flag','next_state',
               'exception_required','exception_type']
leaked = [c for c in target_cols if c in X_train.columns]
if leaked:
    print(f'❌ TARGET LEAKAGE DETECTED: {leaked}')
else:
    print('✅ No target columns leaked into feature matrix')
print(f'Feature matrix columns (first 10): {list(X_train.columns[:10])}')
